# Análise Qualitativa dos Vídeos de Testagem
***

### Importação dos Pacotes Utilitários
***

In [17]:
import pandas as pd
import csv
import cv2
import os

### Funções Utilitárias
***

In [18]:
def image_capture(video_path, time_seconds, output_path):
    '''
    Função que captura um determinado frame do vídeo como imagem.

    Args:
        video_path (str) --> diretório relativo com o vídeo de análise.
        time_seconds (str) --> tempo no formato (00:00), minutos e segundos.
        output_path (str) --> diretório relativo onde o arquivo .png será salvo.
    '''
    
    # abre o vídeo de referência
    cap = cv2.VideoCapture(video_path)

   # verifica se o vídeo foi aberto corretamente
    if not cap.isOpened():
        print("Erro ao abrir o vídeo.")
        return

    # define o tempo em segundos no vídeo para captura
    cap.set(cv2.CAP_PROP_POS_MSEC, time_seconds * 1000)

    # lê o frame no tempo especificado
    ret, frame = cap.read()

    # verifica se o frame especificado foi capturado
    if not ret:
        print("Erro ao ler o quadro do vídeo.")
        return

    # salva a imagem como um arquivo .png
    cv2.imwrite(output_path, frame)

    # libera o objeto de captura
    cap.release()

    print(f"Imagem capturada e salva como {output_path}")

In [19]:
def relative_time(time_str):
    '''
    Converte o formato 00:00, minutos e segundos para o tempo relativo em segundos.

    Args:
        time_str (str) --> tempo no formato (00:00), minutos e segundos.
    Returns:
        time_seconds (int) --> tempo em segundos.
    '''
    
    # separa a string com as informações de minuto e tempo ('00:00')
    minutes, seconds = map(int, time_str.split(':'))
    
    return minutes * 60 + seconds

In [20]:
def create_header(file_name, fieldnames = ['Vídeo', 'Amostra', 'Tempo']):
    '''
    Cria um cabeçalho de uma tabela no formato .csv.

    Args:
        file_name (str) --> caminho relativo com o nome da tabela a ser salva no formato .csv.
        fieldnames (list) --> lista de strings com o cabeçalho das colunas.
    '''

    # abre e cria a tabela no formato .csv
    with open(file_name, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames = fieldnames)
        writer.writeheader()

In [21]:
def add_columns(table_name, line_content, 
                fieldnames = ['Vídeo', 'Amostra', 'Tempo']):
    '''
    Adiciona colunas a uma tabela .csv existente.

    Args:
        table_name (str) --> caminho relativo com o nome da tabela a ser adicionada o conteúdo.
        line_content (list) --> lista contendo o conteúdo das linhas
        fieldnames (list) --> lista de strings com o cabeçalho das colunas.
    '''

    # corrige o formato dos dados a serem adicionados na tabela
    data = list()
    data.append(line_content)

    # cria um dataframe temporário para adicionar o conteúdo da linha no formato especificado
    df_new = pd.DataFrame(data, columns = fieldnames)
    # realizando a leitura do arquivo .csv
    df_old = pd.read_csv(table_name)
    # adicionando a linha a tabela criada
    df_final = pd.concat([df_old, df_new], ignore_index = True)

    # salvar o dataframe final sobrescrito no antigo
    df_final.to_csv(table_name, index = False)


In [22]:
def make_dirs(dir):
    '''
    Cria diretórios e subdiretórios a partir de um caminho especificado.

    Args:
        dir (str) --> caminho relativo do diretório com seus subdiretórios.
    '''
    
    # separar os diretórios e subdiretórios da string
    directorys = dir.split('/')

    # inicializar o caminho atual
    actual_dir = ''

    # iterar pelos diretórios e subdiretórios
    for dir_ref in directorys:
        # atualizar o caminho atual
        actual_dir = os.path.join(actual_dir, dir_ref)

        # criar o diretório atual, se ele ainda não existir
        if not os.path.exists(actual_dir):
            os.makedirs(actual_dir)

In [23]:
def make_frames_dirs(videos):
    '''
    Cria os diretórios de referências para adicionar os frames de análise dos vídeos.

    Args:
        videos (list) --> lista contendo as amostras de frames em segundos para análise em cada um 
                          dos vídeos comparativos.
    '''
    
    # definindo o nome do diretório de referência
    main_dir = 'frame-analise/'
    for i in range(1, len(videos) + 1):
        # criando um subdiretório para armazenar as amostras de frames do vídeo
        video_dir = main_dir + f'video{i}/'
        for j in range(1, len(videos[i - 1]) + 1):
            # salvando os subdiretórios de amostras para cada um dos vídeos
            sample_dir = video_dir + f'amostra{j}/'
            make_dirs(sample_dir)

In [24]:
def return_sample_ref(cenary, video_example):
    '''
    Retorna o caminho relativo contendo o vídeo de exemplo para cada cenário.

    Args:
        cenary (str) --> número do cenário de referência a ser analisado.
        video_example (str) --> vídeo de exemplo do cenário a ser analisado.

    Returns:
        video_dir (str) --> local onde o vídeo está armazenado.
    '''
    
    return f'cenario{cenary}/inferencias/videos/video-exemplo{video_example}.mp4'

In [25]:
def return_output_image(cenary, video_example, sample):
    '''
    Retorna o caminho relativo contendo o local onde a imagen da amostra será salva.

    Args:
        cenary (int) --> cenário de desenvolvimento da análise em questão.
        video_example (int) --> vídeo de exemplo do cenário em questão.
        sample (in) --> amostra de análise do vídeo selecionado.

    Return:
        image_dir (str) --> caminho relativo da imagem de saída salva.
    '''

    return f'frame-analise/video{video_example}/amostra{sample}/cenario{cenary}.png'

### Frames de Referência a Serem Capturados
***

In [26]:
video1 = ['00:08', '00:10', '00:11', '00:14', '00:21', '00:30', '00:36',
          '00:38', '00:43', '00:54', '01:11', '02:00', '02:06']
video2 = ['00:03', '00:08', '00:11', '00:15', '00:17', '00:25', '00:29', 
          '00:40', '00:45', '00:53', '01:00', '01:18']
video3 = ['00:05', '00:09', '00:40', '00:46', '01:17', '01:32', '01:47',
          '02:08', '02:40', '02:52', '03:57', '04:15', '04:53']
video4 = ['00:05', '00:19', '00:31', '00:40', '00:43', '01:20', '01:30', 
          '01:50', '02:02', '02:18', '02:40', '03:23', '04:19', '04:23']
video5 = ['00:06', '00:38', '01:17']
video6 = ['00:05', '00:18', '00:36', '00:38', '00:43', '00:49', '01:01', 
          '01:17',]
video7 = ['00:07', '00:38', '00:54', '01:06']
video8 = ['00:03', '00:13', '00:18', '00:24', '00:47', '00:54', '01:17', 
          '01:31']
video9 = ['00:01', '00:06', '00:08', '00:13', '00:15', '00:19', '00:32', 
          '00:34', '01:15']
video10 = ['00:05', '00:13', '00:28']
video11 = ['00:05', '00:18', '00:26', '00:44', '00:49', '01:30', '02:20', 
           '02:33',]
video12 = ['00:04', '00:12', '00:28', '00:46', '00:49', '01:12', '01:32', 
           '01:38', '01:41']
video13 = ['0:01', '00:05', '00:17']
video14 = ['00:08', '00:42', '00:50', '02:27', '02:33',]

videos = [video1, video2, video3, video4, video5, video6, video7, video8,
          video9, video10, video11, video12, video13, video14]

### Pipeline Geral da Análise dos Frames
***

In [ ]:
# inicializando os diretórios de referências vazios
make_frames_dirs(videos)
# cria uma tabela de referência para adicionar as informações do teste
table_name = 'frame-analise.csv'
create_header(table_name)

for cen in range(1, 4 + 1):
    for i in range(1, len(videos) + 1):
        for j in range(1, len(videos[i - 1]) + 1):
            image_capture(return_sample_ref(cen, i), 
                          relative_time(videos[i - 1][j - 1]), 
                          return_output_image(cen, i, j))
            line_content = [f'{i}', f'{j}', f'{videos[i - 1][j - 1]}']
            add_columns(table_name, line_content) 